# Animating a squall line without a storm track

Notebook 04 moves the camera along a storm's tobac track. This notebook
makes the same kind of animation for a run that has no track. It finds the
storm's position from the model output itself.

The storm is a CM1 squall line. A squall line is a long line of
thunderstorms that moves as one system.

This notebook covers:

1. finding the line at each output time from the surface rain rate;
2. turning those positions into a `sv.Track` that the camera follows;
3. hiding the ends of a line that fills a periodic domain;
4. trying camera positions on single still images;
5. rendering the animation.

Run it in the pixi `blender` environment. Notebook 01 explains the cloud
renders, and notebook 04 explains `sv.BlenderAnimation`. The whole notebook
takes about 30 minutes on a 64-core machine.

In [ ]:
import copy
import functools
import sys
from pathlib import Path

# notebook_setup.py sits next to this notebook. Jupyter starts the kernel
# either in this folder or in the repository root (VS Code's default).
for notebook_folder in (Path.cwd(), Path.cwd() / "examples" / "notebooks"):
    if (notebook_folder / "notebook_setup.py").exists():
        sys.path.insert(0, str(notebook_folder))
        break

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import notebook_setup as setup
import skyvista as sv
from skyvista.cloud_optics import cm1_morrison_hydrometeor_species

## The run

The run covers a 500 x 100 km domain on a 250 m grid. Its boundaries are
periodic on all four sides. Air that leaves one side therefore re-enters
from the opposite side. CM1 wrote one output file every 5 min for 3 h. Each
file holds about 12 GB.

The settings below match the run's namelist, as in notebook 01:

- `ndcnst = 300` cm^-3 sets the fixed cloud droplet number.
- `ihail = 1` makes the rimed ice hail, with a density of 900 kg/m^3.

In [ ]:
SQUALL_LINE_RUN = setup.CM1_SQUALL_LINE_RUN
SQUALL_LINE_OUTPUT_FILES = sorted((SQUALL_LINE_RUN / "output").glob("cm1out_0*.nc"))

# Morrison mixing ratios (kg/kg), number concentrations (#/kg) and the
# air density (kg/m^3), as in notebook 01.
CM1_CLOUD_VARIABLES = ["qc", "qr", "qi", "qs", "qg", "nci", "ncr", "ncs", "ncg", "rho"]

# Morrison settings of this run's namelist: ndcnst = 300 cm^-3, ihail = 1.
squall_line_species = cm1_morrison_hydrometeor_species(
    droplet_number_concentration=300.0e6, graupel_density=900.0
)

# The cloud is resampled onto evenly spaced levels this far apart (m),
# because the run's vertical grid is stretched.
VERTICAL_VOXEL_SPACING = 125.0

# Time between output files (s): the model time of the second file.
with xr.open_dataset(SQUALL_LINE_OUTPUT_FILES[1]) as second_output_file:
    OUTPUT_INTERVAL_SECONDS = float(second_output_file["time"].values[0])


def cm1_output_file(time_seconds):
    '''
    The squall line's output file at a model time.

    CM1 numbers its files from 1 at t = 0, one per output interval, so the
    file number follows from the time.
    '''
    file_number = int(round(time_seconds / OUTPUT_INTERVAL_SECONDS)) + 1
    return SQUALL_LINE_RUN / "output" / f"cm1out_{file_number:06d}.nc"


print(f"{len(SQUALL_LINE_OUTPUT_FILES)} output files, {OUTPUT_INTERVAL_SECONDS:.0f} s apart")

## 1. Finding the line from the surface rain rate

The camera needs one point that moves with the line. The rain under the
line gives that point cheaply.

CM1 writes `prate`, the surface precipitation rate (kg m^-2 s^-1), as a 2-D
field. Reading it at every output time takes about a minute. Reading the
3-D hydrometeor fields instead takes about 40 s per file.

The line's position at each output time is the rain-weighted mean x:

```
line_x = sum(prate * x) / sum(prate)
```

This *rain centroid* lies a little behind the line's leading edge. The
reason is that rain also falls from the trailing stratiform cloud, the wide
layer of lighter rain behind the convective towers. The offset does not
matter here, because the camera only needs a point that moves with the line.

The line fills the whole domain in y. The rain centroid in y therefore says
nothing about the line, so the camera aims at a fixed y instead (section 2).

Output times without rain have no centroid, so the loop skips them.

In [ ]:
# Rain-weighted mean x of the line at each output time with rain.
rain_times_seconds = []
line_x_positions = []
# Rain rate averaged along y, for the time-distance plot below.
along_line_mean_rain_rates = []
for output_path in SQUALL_LINE_OUTPUT_FILES:
    with xr.open_dataset(output_path) as output_file:
        surface_rain_rate = output_file["prate"].isel(time=0).load()
        output_time_seconds = float(output_file["time"].values[0])

    # Skip the times before the first rain reaches the ground.
    total_rain_rate = float(surface_rain_rate.sum())
    if total_rain_rate <= 0.0:
        continue

    # The rain-weighted mean of x over the whole domain.
    rain_weighted_x_sum = float((surface_rain_rate * surface_rain_rate["xh"]).sum())
    rain_times_seconds.append(output_time_seconds)
    line_x_positions.append(rain_weighted_x_sum / total_rain_rate)
    along_line_mean_rain_rates.append(surface_rain_rate.mean("yh").values)

rain_times_seconds = np.array(rain_times_seconds)
line_x_positions = np.array(line_x_positions)
along_line_mean_rain_rates = np.array(along_line_mean_rain_rates)
with xr.open_dataset(SQUALL_LINE_OUTPUT_FILES[0]) as first_output_file:
    domain_x_coordinates = first_output_file["xh"].values

print(f"first rain at t = {rain_times_seconds[0] / 60:.0f} min")

**What to look for.** The figure below is a time-distance plot, also called
a Hovmöller diagram. Its color is the rain rate averaged along the line, in
y. The black curve is the rain centroid. The curve should run through the
middle of the rain band at every time.

In [ ]:
# Convert kg m^-2 s^-1 to mm/h: 1 kg of water over 1 m^2 is 1 mm deep.
SECONDS_PER_HOUR = 3600.0
# The part of the domain the line crosses (km).
PLOTTED_X_RANGE_KM = (100.0, 320.0)

figure, axes = plt.subplots(figsize=(8, 4.5))
rain_rate_mesh = axes.pcolormesh(
    domain_x_coordinates / 1e3,
    rain_times_seconds / 60.0,
    along_line_mean_rain_rates * SECONDS_PER_HOUR,
    cmap="Blues",
    shading="nearest",
)
# The rain centroid: the point the camera follows.
axes.plot(line_x_positions / 1e3, rain_times_seconds / 60.0, color="black", label="rain centroid")
axes.set_xlim(*PLOTTED_X_RANGE_KM)
axes.set_xlabel("x (km)")
axes.set_ylabel("model time (min)")
axes.set_title("Surface rain rate along the line, and its centroid")
axes.legend(loc="upper left")
figure.colorbar(rain_rate_mesh, ax=axes, label="rain rate averaged along y (mm/h)")
plt.show()

In [ ]:
# The line's mean speed: a straight-line fit of position against time.
line_speed, _ = np.polyfit(rain_times_seconds, line_x_positions, deg=1)
print(f"rain centroid: x = {line_x_positions[0] / 1e3:.0f} km at t = {rain_times_seconds[0] / 60:.0f} min")
print(f"rain centroid: x = {line_x_positions[-1] / 1e3:.0f} km at t = {rain_times_seconds[-1] / 60:.0f} min")
print(f"mean eastward speed: {line_speed:.1f} m/s")

**Result.** The rain centroid follows the rain band from the first rain to
the end of the run. The line moves east at a nearly steady 14.5 m/s. It never crosses
the periodic boundary in x, so the centroid needs no special treatment at
the domain edges.

## 2. A track for the camera

`sv.Track` holds a position over time and interpolates between the given
times. Before the first time and after the last, it holds the end
position. The camera therefore stays still until the first rain.

The track's x is the rain centroid. Its y is fixed at the middle of the
line, as section 3 explains.

`smoothing` is the width, in seconds, of a Gaussian average over time. The
centroid jitters a little as rain cells form and decay. Smoothing over
10 min, two output intervals, turns that jitter into a steady glide.

In [ ]:
# The camera aims at this y (m): the middle of three copies of the domain.
LINE_MIDDLE_Y = 150.0e3
# Width of the Gaussian time average of the track (s).
TRACK_SMOOTHING_SECONDS = 600.0

squall_line_track = sv.Track(
    times=rain_times_seconds,
    x=line_x_positions,
    y=np.full_like(line_x_positions, LINE_MIDDLE_Y),
    smoothing=TRACK_SMOOTHING_SECONDS,
)
for check_time_seconds in (1800.0, 5400.0, 10800.0):
    track_x, track_y = squall_line_track.position(check_time_seconds)
    print(f"t = {check_time_seconds / 60:3.0f} min: track at x = {track_x / 1e3:.0f} km")

## 3. Hiding the ends of the line

In the data the line is 100 km long, the width of the domain in y. Rendered
as it is, the line would end in two flat walls at y = 0 and y = 100 km.
The real line has no such ends. They exist only because the domain is
periodic.

`tile_periodic_in_y` lays three copies of the domain end to end, so the
line becomes 300 km long. The copies join without a seam because the
northern edge of each copy continues into the southern edge of the next.
That is what periodic boundaries mean. The camera aims at the middle copy,
so neither end of the line comes into view.

`build_squall_line_frame` also crops each frame in x around the track. The
crop window moves with the line. It keeps 100 km behind the rain centroid,
for the trailing stratiform cloud. It keeps 80 km ahead, for the forward
anvil. Less data makes the export faster.

In [ ]:
# The crop window around the rain centroid (m): behind it, then ahead of it.
CROP_DISTANCE_BEHIND_LINE = 100.0e3
CROP_DISTANCE_AHEAD_OF_LINE = 80.0e3
# Copies of the domain laid end to end in y.
PERIODIC_Y_COPIES = 3


def tile_periodic_in_y(clouds, n_copies):
    '''
    Lay ``n_copies`` of a y-periodic domain end to end.

    Each copy's y coordinates continue where the previous copy's end, one
    grid spacing on, so the tiled grid stays evenly spaced. The result is
    exact only for a run with periodic north and south boundaries, where
    the field at the northern edge continues into the southern edge.
    '''
    y_coordinates = clouds["y"].values
    # The domain's length: the span of the cell centers plus one grid cell.
    y_grid_spacing = float(y_coordinates[1] - y_coordinates[0])
    domain_length_y = float(y_coordinates[-1] - y_coordinates[0]) + y_grid_spacing
    # Each copy shifted north by a whole number of domain lengths.
    shifted_copies = [
        clouds.assign_coords(y=y_coordinates + copy_index * domain_length_y)
        for copy_index in range(n_copies)
    ]
    return xr.concat(shifted_copies, dim="y")


# Keep the last few times in memory, about 4 GB each, so that repeated
# still images of one time (section 4) skip the 40 s read.
@functools.lru_cache(maxsize=4)
def load_squall_line_clouds(time_seconds):
    '''
    The hydrometeors at one time, cropped around the line and tiled in y.

    The crop window follows the smoothed track, so the line stays in the
    middle of the exported volume as it moves east.
    '''
    track_x, _ = squall_line_track.position(time_seconds)
    with xr.open_dataset(cm1_output_file(time_seconds)) as output_file:
        # CM1's scalar-point coordinates are xh, yh, zh, already in meters.
        cropped_clouds = (
            output_file[CM1_CLOUD_VARIABLES]
            .isel(time=0)
            .rename({"xh": "x", "yh": "y", "zh": "z"})
            .sel(x=slice(track_x - CROP_DISTANCE_BEHIND_LINE, track_x + CROP_DISTANCE_AHEAD_OF_LINE))
            .load()
        )
    return tile_periodic_in_y(cropped_clouds, PERIODIC_Y_COPIES)


def build_squall_line_frame(time_seconds):
    '''The scene at one time: the line's clouds, cropped and tiled.'''
    return sv.Scene().add_clouds(
        load_squall_line_clouds(time_seconds),
        species=squall_line_species,
        vertical_spacing=VERTICAL_VOXEL_SPACING,
    )

## 4. Trying camera positions on still images

The full animation takes about 20 minutes. A single still image takes about
a minute. Choose the camera on still images first.

The camera follows a `sv.CameraPath`, a list of `sv.CameraKeyframe`s at
model times. Notebook 04 describes the keyframe quantities. Each one is
relative to the track:

- `azimuth_deg`: the direction from the track to the camera. 0 is east and
  -90 is south.
- `horizontal_distance`: the camera's distance from the track (m).
- `height`: the camera's height above the ground (m).
- `target_height`: the height of the point the camera aims at, above the
  track (m).
- `lens_mm`: the focal length. A smaller value gives a wider view.

A keyframe can leave a quantity out. The camera path then interpolates that
quantity through the keyframes that set it.

`render_test_still(time, camera_path, name)` renders one time with the
camera that `camera_path` gives at that time. It places the camera with
`camera_path.at(time)`, which is the same call the animation uses. A still
that looks right therefore gives the same frame in the animation.

The export settings come first. The sun is in the south-west, 30 degrees
above the horizon. Every frame must share one origin (notebook 04).

In [ ]:
# Frame size of the stills and the preview animation (px).
PREVIEW_RESOLUTION = (640, 360)

squall_line_config = sv.BlenderExportConfig.for_clouds(quality="preview")
squall_line_config.render.resolution = PREVIEW_RESOLUTION
squall_line_config.world.sun_azimuth_deg = -150.0
squall_line_config.world.sun_elevation_deg = 30.0
# One fixed origin for every frame: the domain's south-west corner.
squall_line_config.transform.origin_shift = (0.0, 0.0, 0.0)


def render_test_still(time_seconds, camera_path, still_name):
    '''
    Render one model time with the camera a camera path gives at that time.

    The camera is fixed from ``camera_path.at(time_seconds)``: an explicit
    location, aim point and lens in model coordinates. BlenderAnimation
    places the camera of each frame with the same call, so the still
    matches the animation frame at this time.

    Returns:
        The rendered image.
    '''
    camera_state = camera_path.at(time_seconds)
    still_config = copy.deepcopy(squall_line_config)
    still_config.camera.location = camera_state.location
    still_config.camera.look_at = camera_state.look_at
    still_config.camera.lens_mm = camera_state.lens_mm

    still_bundle_dir = setup.WORK_DIR / "squall_line_stills" / still_name
    build_squall_line_frame(time_seconds).to_blender(
        still_bundle_dir, config=still_config, render=True, quiet=True
    )
    return setup.rendered_frame(still_bundle_dir)

**Try one view.** The cell below renders the line at 3 h from a single
keyframe. Change any value and run the cell again. A camera path with one
keyframe holds its values at every time, so the keyframe's own time does
not matter here.

In [ ]:
TRIAL_TIME_SECONDS = 10800.0

trial_camera_path = sv.CameraPath(
    [
        sv.CameraKeyframe(
            TRIAL_TIME_SECONDS,
            azimuth_deg=-30.0,
            horizontal_distance=100.0e3,
            height=10000.0,
            target_height=6000.0,
            lens_mm=24.0,
        )
    ],
    target=squall_line_track,
)
trial_still_path = render_test_still(TRIAL_TIME_SECONDS, trial_camera_path, "trial")
setup.show_image(trial_still_path, width=800)

**Check the whole path.** Once single views look right, write them as
keyframes at their own times. The cell after the keyframes renders a still
at each check time and lays the stills out as a contact sheet. Include
times between keyframes, because the interpolated cameras can surprise.

The path below starts close, while the young towers are small. It then
pulls back as the line's forward anvil spreads. By 3 h the anvil reaches
about 70 km ahead of the line along the line's whole length. A camera
closer than that to the line, on its east side, sits under the anvil and
sees only the anvil's dark base. The last keyframe therefore puts the
camera about 87 km east of the line, beyond the anvil's edge.

In [ ]:
squall_line_camera_path = sv.CameraPath(
    [
        # Young towers, about 8 km tall: from above, close in.
        sv.CameraKeyframe(
            1800.0, azimuth_deg=-60.0, horizontal_distance=35.0e3,
            height=8000.0, target_height=1500.0, lens_mm=24.0,
        ),
        # The anvil starts to spread ahead of the line.
        sv.CameraKeyframe(5400.0, horizontal_distance=80.0e3, height=8000.0, target_height=4000.0),
        # The mature line: beyond the edge of a forward anvil about 70 km wide.
        sv.CameraKeyframe(
            10800.0, azimuth_deg=-30.0, horizontal_distance=100.0e3,
            height=10000.0, target_height=6000.0,
        ),
    ],
    target=squall_line_track,
)

In [ ]:
# Model times of the check stills (s): the keyframes and between them.
CHECK_TIMES_SECONDS = [1800.0, 3600.0, 5400.0, 8100.0, 10800.0]

check_still_paths = [
    render_test_still(check_time_seconds, squall_line_camera_path, f"check_{check_time_seconds:.0f}s")
    for check_time_seconds in CHECK_TIMES_SECONDS
]

# The stills as one contact sheet, labelled with their model times.
check_sheet_path = setup.WORK_DIR / "squall_line_check_sheet.png"
sv.compose.Figure(
    panels=[
        sv.compose.Panel(still_path, [sv.compose.Text(f"t = {check_time_seconds / 60:.0f} min", size=0.08)])
        for still_path, check_time_seconds in zip(check_still_paths, CHECK_TIMES_SECONDS, strict=True)
    ],
    n_columns=2,
).render(check_sheet_path)
setup.show_image(check_sheet_path, width=1000)

## 5. The animation

The animation uses every output time from 30 min to 3 h, 5 min apart. That
gives 31 frames. At 4 frames per second, the video lasts about 8 s.

The output is only 5 min apart, so the clouds jump between frames. The
camera moves smoothly between output times, but the cloud data cannot.

In [ ]:
# Every output time from the first keyframe to the end of the run (s).
ANIMATION_START_SECONDS = 1800.0
ANIMATION_END_SECONDS = 10800.0
# Video frames per second.
ANIMATION_FPS = 4.0

animation_times_seconds = np.arange(
    ANIMATION_START_SECONDS, ANIMATION_END_SECONDS + 1.0, OUTPUT_INTERVAL_SECONDS
)

# A title and the model time on every frame.
squall_line_figure = sv.compose.Figure(
    panels=[
        sv.compose.Panel(
            lambda frame: frame.render_path,
            [
                sv.compose.Text("Squall line (CM1, 250 m grid)", size=0.06),
                sv.compose.TimeStamp(anchor="lower left", size=0.06),
            ],
        )
    ]
)

squall_line_animation = sv.BlenderAnimation(
    setup.WORK_DIR / "squall_line_animation",
    times=animation_times_seconds,
    build_frame=build_squall_line_frame,
    config=squall_line_config,
    camera_path=squall_line_camera_path,
    fps=ANIMATION_FPS,
)
squall_line_video = squall_line_animation.run(
    figure=squall_line_figure, export_workers=8, render_processes=4
)

In [ ]:
# Every sixth composed frame, as one contact sheet: 30 min apart.
CONTACT_SHEET_FRAME_STEP = 6

animation_sheet_path = setup.WORK_DIR / "squall_line_animation_sheet.png"
composed_frame_paths = sorted(squall_line_animation.composed_dir.glob("frame_*.png"))
sv.compose.Figure(
    panels=[sv.compose.Panel(frame_path) for frame_path in composed_frame_paths[::CONTACT_SHEET_FRAME_STEP]],
    n_columns=2,
).render(animation_sheet_path)
setup.show_image(animation_sheet_path, width=1000)

In [ ]:
setup.show_video(squall_line_video)

## From preview to the full animation

Three settings turn this preview into a final video:

- `squall_line_config.render.set_quality("final")`: 128 samples per pixel
  at 1920 x 1080 instead of 16 at 640 x 360.
- `run(render_processes=...)`: several Blender processes share the frames.
  This uses many cores better than one large process.
- A new `output_dir` for the `BlenderAnimation`. Every stage skips frames
  already on disk, so the old directory would keep the preview frames.

The same steps work for any run without a track. Pick a 2-D field that
marks the storm, such as rain rate or column-maximum vertical velocity.
Take its weighted centroid at each output time, and pass the centroids to
`sv.Track`.